# Full Project of brochure making a week 1 project from the course

we will use our website summurizer and make one more call to the llm asking to act as professional brochure maker and to make a brochure from colected data

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [8]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5.4-mini'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## Generating System Prompt

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

### Sorting relevant links for brochure from all the links

new arg learnt response format - which specifys in what format do we get response in

In [9]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [10]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'posts page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'profile page', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [11]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [12]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5.4-mini
Found 11 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'posts page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'avatar page', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [13]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5.4-mini
Found 14 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'docs page', 'url': 'https://huggingface.co/docs'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'company page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'github page', 'url': 'https://github.com/huggingface'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'linkedin page',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5.4-mini

In [14]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [15]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5.4-mini
Found 16 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
empero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF
Updated
7 days ago
•
1.46M
•
1.48k
zai-org/GLM-5.2
Updated
3 days ago
•
209k
•
3.41k
baidu/Unlimited-OCR
Updated
2 days ago
•
988k
•
1.72k
deepreinforce-ai/Ornith-1.0-35B-GGUF
Updated
10 days ago
•
360k
•
713
nvidia/Qwen3.6-27B-NVFP4
Updated
5 day

### Defining character

Including a single work like humorous can change response drastically

In [ ]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [16]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [17]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5.4-mini
Found 13 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nempero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF\nUpdated\n7 days ago\n•\n1.46M\n•\n1.48k\nzai-org/GLM-5.2\nUpdated\n3 days ago\n•

In [23]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [24]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5.4-mini
Found 15 relevant links


# Hugging Face

## Overview
Hugging Face is **the AI community building the future**. It provides a collaboration platform where the machine learning community can create, discover, and work together on:

- **Models**
- **Datasets**
- **Applications / Spaces**
- **Tools and APIs**

With a focus on openness and collaboration, Hugging Face positions itself as the home of machine learning for builders, researchers, and teams.

## What they offer
Hugging Face’s platform includes:

- **The Hub** for hosting Git-based models, datasets, and Spaces
- **Spaces** for building and sharing AI applications
- **Datasets** for finding and publishing data
- **Inference services** such as Inference Providers and Inference Endpoints
- **Storage Buckets** for managing assets
- **Docs, CLI, Python, and JavaScript libraries** to help developers integrate and build

The ecosystem is designed to help users move faster and collaborate better across the full ML lifecycle.

## Community and culture
Hugging Face presents itself as a **community-first company**. Its culture is rooted in:

- Open collaboration
- Sharing models and datasets publicly
- Supporting both individuals and organizations
- Building tools for researchers, developers, and enterprise teams
- Encouraging participation through community channels like **Discord, the Forum, GitHub, Blog, Posts, and Daily Papers**

This gives the company a strong identity as both a platform and a community hub.

## Customers and users
Hugging Face serves a broad range of users, including:

- **ML researchers** who publish and explore models and datasets
- **Developers** building AI apps and experiences
- **Enterprises** looking for production AI infrastructure and support
- **Organizations and teams** collaborating on shared AI work
- **The wider AI community** discovering and running applications in Spaces

Its platform supports everything from experimental projects to enterprise deployments.

## Careers and recruiting appeal
While no specific jobs were listed on the pages provided, Hugging Face appears attractive to people who want to:

- Work on modern AI infrastructure
- Contribute to open and collaborative ML tools
- Build products used by a large global technical community
- Help shape the future of accessible AI development

For recruits, the company’s mission-driven, community-centered positioning suggests a strong fit for builders who value openness and impact.

## Why it stands out
Hugging Face stands out because it combines:

- A massive repository of **models and datasets**
- A place to launch **AI applications**
- Developer-friendly **libraries and docs**
- A strong **community and ecosystem**
- Support for both **individual creators and enterprise customers**

## In short
Hugging Face is a leading AI platform where the machine learning community collaborates, ships applications, and shares the building blocks of modern AI. It is well suited for customers who want to develop faster, researchers who want to share widely, and talent who want to help build the future of AI.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation


learnt about streaming response in a typewriter tyoe animation giving a better user experience

### Key takeouts

new arg - stream = True is False by default

we need empty response and md to start with then we update it for every chuck we get in stream

the response is stored in chuck.choices[0].delta.content instead normal messages.content

Also we need to assign a display id for empty markdown to further update the same markdown 

In [26]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [27]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5.4-mini
Found 16 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Hugging Face

**The AI community building the future.**

Hugging Face is a collaboration platform for the machine learning community, where people build, share, and discover **models, datasets, and applications**. It serves as a central hub for AI work, helping teams and individuals move faster from research to real-world use.

## What Hugging Face Offers

- **Models:** Access to a huge ecosystem of ML models, with **2M+ models** available to browse.
- **Datasets:** A growing library of data resources, with **500k+ datasets** to explore.
- **Spaces:** A place to host and run AI applications and demos, with **1M+ applications**.
- **Buckets:** Storage options for working with AI assets and workflows.
- **Docs and developer tools:** Guides, references, APIs, CLI tools, Python and JavaScript libraries to support development.

## Why People Use It

Hugging Face is designed for collaboration and speed. Its platform helps the ML community:

- host and collaborate on public models, datasets, and applications
- discover trending AI projects
- build and deploy AI apps more easily
- connect through a broad ecosystem of docs, community channels, and tools

## Community and Culture

Hugging Face emphasizes openness and community. Its brand message centers on the idea of **the AI community building the future**, and its products reflect a collaborative approach to machine learning. The company supports engagement through:

- **Discord**
- **Forum**
- **GitHub**
- **Blog, Posts, Daily Papers, and Learn**

This suggests a culture focused on sharing knowledge, experimentation, and community-driven progress.

## Customers and Use Cases

Hugging Face appears to serve:

- **ML developers and researchers**
- **startups and builders launching AI apps**
- **enterprises** needing support, endpoints, and infrastructure
- **organizations** managing AI workflows and collaboration

Example use cases highlighted on the platform include:

- voice chat applications
- image editing tools
- video generation from images and prompts
- OCR for extracting text from images and PDFs

## Careers / Talent

While no specific job listings are shown here, Hugging Face’s positioning suggests a workplace for people who care about:

- machine learning and AI infrastructure
- open collaboration
- developer tools and platform engineering
- research-to-product workflows

Its ecosystem and community-first approach may especially appeal to engineers, researchers, and product builders interested in shaping the future of AI.

## In Short

Hugging Face is a leading AI collaboration platform that brings together models, datasets, and applications in one place. For customers, it offers tools to build and deploy faster. For investors, it shows strong platform-scale potential. For recruits, it signals a mission-driven company rooted in community, openness, and practical AI innovation.

In [29]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5.4-mini
Found 14 relevant links


# Hugging Face: the AI community with a smile

Hugging Face is basically the **grand central station of machine learning** — where models, datasets, and applications all meet up, collaborate, and occasionally outshine each other in public.

## What they do
At its core, Hugging Face is:

- a platform for **sharing and discovering AI models**
- a home for **datasets**
- a launchpad for **AI apps** through **Spaces**
- a place for teams and companies to build faster with **Enterprise**, **Inference Endpoints**, **Storage Buckets**, and more

In short: if AI had a social network, a toolbox, and a coworking space, this would be it.

## Why people use it
Hugging Face says it plainly:  
**“The AI community building the future.”**

That community focus shows up everywhere:

- **2M+ models** to browse
- **500K+ datasets**
- **1M+ applications**
- public collaboration for builders, researchers, and product teams

It’s less “closed fortress of secret sauce” and more “open lab with excellent indexing.”

## A few popular attractions
The platform showcases a busy ecosystem of trending models and Spaces, including:

- **HF Realtime Voice** — voice chat over WebSocket
- **Pro Realism Edit Studio** — image editing with one or two input images
- **Unlimited OCR** — extract text from images and PDFs
- video generation tools, speech tools, and more

Basically: if your AI wish list says “can it also do that?” the answer is often “yes, and there’s a Space for it.”

## Community and culture
Hugging Face is built around collaboration. Their community includes:

- individual contributors
- organizations
- companies
- universities
- non-profits

They also lean heavily into open sharing, which gives the vibe of a very smart group project where everyone actually submitted their part on time.

## Who’s there
Some notable organizations on the platform include:

- **DeepSeek**
- **Qwen**
- **Meta Llama**
- and Hugging Face’s own team

These groups collectively signal one thing: this is where serious AI builders come to hang out, publish, compare notes, and occasionally dominate leaderboards.

## For customers
If you’re a company looking to build with AI, Hugging Face offers:

- a huge library of models and datasets
- enterprise options
- infrastructure for inference and storage
- collaboration tools for teams of all sizes

So whether you’re prototyping a clever app or deploying at scale, Hugging Face is trying to be your AI utility belt.

## For recruits
If you like:

- open source energy
- fast-moving AI work
- community-driven building
- the idea that “model zoo” is a business strategy

…then Hugging Face looks like a fun place to work, because the whole company revolves around making AI more accessible, collaborative, and usable.

## The bottom line
Hugging Face is the place where machine learning stops being mysterious wizardry and starts being shared infrastructure.

Or, to put it less formally:  
**they’re building the future of AI — and making it browseable.**

# All the code belongs to Ed i have no ownership just documenting my progress